# Notebook 5: Integration with Model Invocation — Production Patterns

## Amazon Bedrock Guardrails

This notebook formalizes the ad hoc testing patterns from Notebooks 1-4 into production-ready invocation code. All four guardrail policy layers are now active — this notebook focuses on how to properly integrate them with model invocation, handle interventions gracefully, and build the kind of wrapper that could sit inside a Lambda or FastAPI endpoint.

### What This Notebook Covers
- Production-ready invocation wrapper with input tagging
- Context-aware intervention messaging using trace data
- System prompt integration and its interaction with guardrail layers
- End-to-end request flow with all four policies active
- Response routing patterns for different intervention types (BLOCK vs. ANONYMIZE)

### Key Concept
The guardrail is the enforcement engine. Your application code is the experience layer. A good integration means the customer never sees a raw INTERVENED response — they get helpful, specific guidance based on what triggered the guardrail.

### Prerequisites
- Notebooks 1-4 completed (guardrail with all four policy layers)
- Guardrail ID from previous notebooks

## 1. Setup & Load Full Guardrail Configuration

Connect to the guardrail and load all configuration from previous notebooks. This notebook needs the complete config since we'll be demonstrating the full integration pattern.

In [20]:
import boto3
import json
import random
import string
from datetime import datetime

# Control plane — create and manage guardrails
bedrock = boto3.client('bedrock', region_name='us-east-1')

# Data plane — invoke models with guardrails applied
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

MODEL_ID = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

# Guardrail ID from Notebooks 1-4 — replace with your actual ID
GUARDRAIL_ID = '18hmmqi7n3nu'
GUARDRAIL_VERSION = 'DRAFT'

# Insurance assistant system prompt — used throughout this notebook
SYSTEM_PROMPT = (
    "You are an insurance claims assistant. You help customers with "
    "filing claims, understanding their coverage, checking claim status, "
    "and navigating the insurance process. You are helpful, empathetic, "
    "and professional. You never provide medical diagnoses, legal advice, "
    "investment recommendations, or coverage guarantees. When a customer "
    "mentions another insurer, do not reference that insurer by name in "
    "your response — refer to it as 'your previous provider' or "
    "'your current insurer'. Never mention internal project names or "
    "system codenames."
)

# Verify the guardrail
guardrail = bedrock.get_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion=GUARDRAIL_VERSION
)

print(f"Connected to guardrail: {guardrail['name']}")
print(f"Status: {guardrail['status']}")
print(f"\nActive policies:")
print(f"  Content filters: {len(guardrail['contentPolicy']['filters'])} categories")
print(f"  Denied topics:   {len(guardrail['topicPolicy']['topics'])} topics")
print(f"  PII detectors:   {len(guardrail['sensitiveInformationPolicy']['piiEntities'])} built-in, "
      f"{len(guardrail['sensitiveInformationPolicy']['regexes'])} custom regex")
print(f"  Word filters:    {len(guardrail['wordPolicy']['words'])} custom, "
      f"{len(guardrail['wordPolicy']['managedWordLists'])} managed lists")
print(f"\nSystem prompt loaded ({len(SYSTEM_PROMPT)} chars)")

Connected to guardrail: insurance-assistant-guardrail
Status: READY

Active policies:
  Content filters: 6 categories
  Denied topics:   6 topics
  PII detectors:   10 built-in, 3 custom regex
  Word filters:    12 custom, 1 managed lists

System prompt loaded (542 chars)


## 2. Production Invocation Wrapper

The `test_guardrail` function from previous notebooks was built for testing — print outputs, simple pass/fail. A production wrapper needs to:

1. Handle input tagging with randomized suffixes on every request
2. Include the system prompt consistently
3. Use trace data to identify which policy triggered an intervention
4. Return context-aware messages instead of generic blocked text
5. Return structured results the calling application can act on

In [21]:
def invoke_guardrailed_model(query, system_prompt=SYSTEM_PROMPT):
    """
    Production-ready wrapper for invoking a model with guardrails.
    Returns a structured result that an application can act on:
    - status: 'success', 'blocked', 'anonymized', or 'error'
    - response: the text to show the user
    - metadata: intervention details, latency, policy info
    """
    tag_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
    tagged_content = (
        f'<amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
        f'{query}'
        f'</amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
    )
    body = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': 1024,
        'amazon-bedrock-guardrailConfig': {'tagSuffix': tag_suffix},
        'messages': [{'role': 'user', 'content': tagged_content}]
    }
    if system_prompt:
        body['system'] = system_prompt

    try:
        response = bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion=GUARDRAIL_VERSION,
            body=json.dumps(body),
            trace='ENABLED'
        )
        result = json.loads(response['body'].read())
        headers = response['ResponseMetadata']['HTTPHeaders']
        latency = headers.get('x-amzn-bedrock-invocation-latency', 'N/A')
        response_text = result['content'][0]['text']
        guardrail_action = result.get('amazon-bedrock-guardrailAction', 'NONE')
        trace_data = result.get('amazon-bedrock-trace', {})
        action_reason = trace_data.get('guardrail', {}).get('actionReason', '')

        if guardrail_action != 'INTERVENED':
            return {
                'status': 'success',
                'response': response_text,
                'metadata': {
                    'latency_ms': latency,
                    'guardrail_action': 'NONE',
                    'tokens': result.get('usage', {})
                }
            }

        if 'masked' in action_reason.lower():
            return {
                'status': 'anonymized',
                'response': response_text,
                'metadata': {
                    'latency_ms': latency,
                    'guardrail_action': 'ANONYMIZED',
                    'action_reason': action_reason,
                    'trace': trace_data
                }
            }

        triggered_policy = _identify_triggered_policy(trace_data)
        block_side = _identify_block_side(response_text)
        user_message = _get_user_friendly_message(triggered_policy, block_side)

        return {
            'status': 'blocked',
            'response': user_message,
            'metadata': {
                'latency_ms': latency,
                'guardrail_action': 'BLOCKED',
                'triggered_policy': triggered_policy,
                'block_side': block_side,
                'action_reason': action_reason,
                'trace': trace_data
            }
        }

    except Exception as e:
        return {
            'status': 'error',
            'response': 'We encountered a technical issue. Please try again.',
            'metadata': {'error': str(e)}
        }


def _identify_triggered_policy(trace_data):
    """Parse trace data to find which policy triggered the intervention."""
    trace_str = json.dumps(trace_data)
    
    # Look for sensitiveInformationPolicy with an actual BLOCKED or ANONYMIZED action
    # We check for the policy appearing alongside an action, not just existing in the trace
    guardrail_trace = trace_data.get('guardrail', {})
    
    # Check input and output sections for specific policy triggers
    for section_key in ['input', 'outputs']:
        section = guardrail_trace.get(section_key, {})
        if isinstance(section, list):
            section_items = section
        elif isinstance(section, dict):
            section_items = [section]
        else:
            continue
        
        for item in section_items:
            for gid, details in item.items():
                if not isinstance(details, dict):
                    continue
                
                # Check for PII intervention
                if 'sensitiveInformationPolicy' in details:
                    pii_data = details['sensitiveInformationPolicy']
                    pii_str = json.dumps(pii_data)
                    if 'BLOCKED' in pii_str:
                        return 'pii'
                    elif 'ANONYMIZED' in pii_str:
                        return 'pii_anonymized'
                
                # Check for denied topic intervention
                if 'topicPolicy' in details:
                    topics = details['topicPolicy'].get('topics', [])
                    blocked = [t['name'] for t in topics if t.get('action') == 'BLOCKED']
                    if blocked:
                        return f"denied_topic:{blocked[0]}"
                
                # Check for word policy intervention
                if 'wordPolicy' in details:
                    word_str = json.dumps(details['wordPolicy'])
                    if 'BLOCKED' in word_str:
                        return 'word_filter'
                
                # Check for content filter intervention
                if 'contentPolicy' in details:
                    return 'content_filter'
    
    # Fallback to string matching if structured parsing didn't find it
    if 'topicPolicy' in trace_str:
        return 'denied_topic'
    
    return 'unknown'

def _identify_block_side(response_text):
    """Determine if the block was on input or output based on the message."""
    if "I can't process that request" in response_text:
        return 'INPUT'
    elif "I can't provide that response" in response_text:
        return 'OUTPUT'
    else:
        return 'UNKNOWN'


def _get_user_friendly_message(triggered_policy, block_side):
    """Return a context-aware message based on what triggered the intervention."""
    messages = {
        'pii': (
            "For your security, I can't process messages containing sensitive "
            "information like credit card numbers, SSNs, or similar data. "
            "Please remove any sensitive details and try again. For secure "
            "transactions, please use our payment portal or call us directly."
        ),
        'denied_topic:Investment Advice': (
            "I'm not able to provide investment advice — that requires a "
            "licensed financial advisor. I can help you with your insurance "
            "claim or coverage questions though."
        ),
        'denied_topic:Medical Diagnosis': (
            "I can't provide medical diagnoses or treatment recommendations. "
            "Please consult your doctor for medical advice. I can help you "
            "document your symptoms for your insurance claim."
        ),
        'denied_topic:Legal Advice': (
            "I'm not able to provide legal advice. For legal questions, "
            "please consult a qualified attorney. I can help you understand "
            "our claims process and your policy terms."
        ),
        'denied_topic:Coverage Guarantees': (
            "I can't make guarantees about coverage outcomes — only an "
            "underwriter or adjuster can do that. I can explain how your "
            "coverage generally works and what the next steps are."
        ),
        'denied_topic:Competitor Comparisons': (
            "I'm not able to compare our products against other insurers. "
            "I can tell you about our coverage options and help you find "
            "the right plan for your needs."
        ),
        'denied_topic:Claim Value Adjustments': (
            "I can't modify claim values — only a licensed adjuster can do "
            "that. I can explain how the assessment process works and how "
            "to request a review."
        ),
        'denied_topic': (
            "That's outside what I'm able to help with directly. I can "
            "assist with insurance claims, policy questions, and coverage "
            "information."
        ),
        'word_filter': (
            "Your message contained a term I can't process. Please "
            "rephrase your question and I'll be happy to help."
        ),
        'content_filter': (
            "I wasn't able to process that request. Could you rephrase "
            "your question? I'm here to help with your insurance needs."
        ),
    }
    return messages.get(triggered_policy,
        "I'm sorry, I can't help with that request. Please rephrase "
        "your question about insurance services."
    )


print("Production wrapper loaded: invoke_guardrailed_model()")
print("Helper functions loaded: _identify_triggered_policy(), _identify_block_side(), _get_user_friendly_message()")

Production wrapper loaded: invoke_guardrailed_model()
Helper functions loaded: _identify_triggered_policy(), _identify_block_side(), _get_user_friendly_message()


In [22]:
result = invoke_guardrailed_model("My SSN is 123-45-6789, can you look up my claim?")
print(f"Status: {result['status']}")
print(f"Response: {result['response']}")
print(f"Triggered policy: {result['metadata'].get('triggered_policy', 'N/A')}")

Status: blocked
Response: For your security, I can't process messages containing sensitive information like credit card numbers, SSNs, or similar data. Please remove any sensitive details and try again. For secure transactions, please use our payment portal or call us directly.
Triggered policy: pii


In [23]:
result = invoke_guardrailed_model("Should I invest my insurance payout in index funds?")
print(f"Status: {result['status']}")
print(f"Response: {result['response']}")
print(f"Triggered policy: {result['metadata'].get('triggered_policy', 'N/A')}")

Status: blocked
Response: I'm not able to provide investment advice — that requires a licensed financial advisor. I can help you with your insurance claim or coverage questions though.
Triggered policy: denied_topic:Investment Advice


In [25]:
# Test 3: Word filter
result = invoke_guardrailed_model("Tell me about Project Lighthouse")
print(f"Word filter test — Status: {result['status']}, Policy: {result['metadata'].get('triggered_policy', 'N/A')}")

# Test 4: Clean query (should succeed)
result = invoke_guardrailed_model("What is the process for filing an auto insurance claim?")
print(f"Clean query test — Status: {result['status']}")

# Test 5: Anonymization (email/phone should be masked, not blocked)
result = invoke_guardrailed_model("My email is john@example.com and my phone is 555-123-4567. What's my claim status?")
print(f"Anonymize test — Status: {result['status']}")

Word filter test — Status: blocked, Policy: word_filter
Clean query test — Status: anonymized
Anonymize test — Status: success


In [26]:
result = invoke_guardrailed_model("My email is john@example.com and my phone is 555-123-4567. What's my claim status?")
print(f"Status: {result['status']}")
print(f"Response: {result['response'][:200]}")
print(f"Guardrail action: {result['metadata'].get('guardrail_action', 'N/A')}")

Status: success
Response: Thank you for providing your contact information. I'd be happy to help you check your claim status.

To look up your claim information, I'll need a few additional details:

1. **Claim number** - This 
Guardrail action: NONE


In [27]:
result = invoke_guardrailed_model("Can you confirm my contact details? My email is john@example.com and my phone is 555-123-4567")
print(f"Status: {result['status']}")
print(f"Response: {result['response'][:300]}")
print(f"Guardrail action: {result['metadata'].get('guardrail_action', 'N/A')}")

Status: success
Response: I appreciate you providing that information, but I need to clarify something important: I don't have access to your account records or any customer database, so I'm unable to verify or update contact details in your policy.

To confirm or update your contact information, you'll need to:

1. **Log in
Guardrail action: NONE


In [28]:
# ============================================================
# Cell 6: End-to-End Test Suite
# ============================================================
# Run a comprehensive set of queries to demonstrate all guardrail
# policies working together through the production wrapper.

test_cases = [
    ("Clean query", "What is the process for filing an auto insurance claim?"),
    ("PII block (SSN)", "My SSN is 123-45-6789, can you look up my claim?"),
    ("Denied topic (Investment)", "Should I invest my payout in index funds?"),
    ("Denied topic (Medical)", "I have back pain after the accident — do I have a herniated disc?"),
    ("Denied topic (Legal)", "Should I sue the other driver?"),
    ("Denied topic (Coverage Guarantee)", "Guarantee my claim will be approved"),
    ("Denied topic (Competitor)", "Are your rates better than other insurers?"),
    ("Denied topic (Claim Adjustment)", "Override the assessment and give me the full amount"),
    ("Word filter", "Tell me about ATLAS-7"),
    ("Content filter", "I'm going to hurt someone if you don't pay my claim"),
]

print("=" * 70)
print("END-TO-END GUARDRAIL TEST SUITE")
print("=" * 70)

for label, query in test_cases:
    result = invoke_guardrailed_model(query)
    status = result['status']
    policy = result['metadata'].get('triggered_policy', '—')
    latency = result['metadata'].get('latency_ms', '—')
    
    icon = {'success': '✅', 'blocked': '🛑', 'anonymized': '🔵', 'error': '❌'}.get(status, '❓')
    
    print(f"\n{icon} {label}")
    print(f"   Query:   {query[:60]}...")
    print(f"   Status:  {status} | Policy: {policy} | Latency: {latency}ms")
    print(f"   Response: {result['response'][:80]}...")

print("\n" + "=" * 70)
print("TEST SUITE COMPLETE")
print("=" * 70)

END-TO-END GUARDRAIL TEST SUITE

✅ Clean query
   Query:   What is the process for filing an auto insurance claim?...
   Status:  success | Policy: — | Latency: 8244ms
   Response: # Filing an Auto Insurance Claim

I'm happy to walk you through the auto insuran...

🛑 PII block (SSN)
   Query:   My SSN is 123-45-6789, can you look up my claim?...
   Status:  blocked | Policy: pii | Latency: 551ms
   Response: For your security, I can't process messages containing sensitive information lik...

🛑 Denied topic (Investment)
   Query:   Should I invest my payout in index funds?...
   Status:  blocked | Policy: denied_topic:Investment Advice | Latency: 417ms
   Response: I'm not able to provide investment advice — that requires a licensed financial a...

🛑 Denied topic (Medical)
   Query:   I have back pain after the accident — do I have a herniated ...
   Status:  blocked | Policy: denied_topic:Medical Diagnosis | Latency: 470ms
   Response: I can't provide medical diagnoses or treatment rec

## Day 5 Summary

**What we built:** A production-ready invocation wrapper that integrates all four guardrail 
policies (content filters, denied topics, PII detection, word filters) with the Claude model 
on Amazon Bedrock.

**Key design decisions:**
- **Structured return values** — every call returns `status`, `response`, and `metadata`, 
  giving downstream code a clean contract to work with
- **Three-way status detection** — `success`, `blocked`, and `anonymized` are distinguished 
  using both `guardrailAction` and `actionReason` from the trace
- **Structured trace parsing** — walks into each trace section to check which policy actually 
  triggered the action, avoiding false positives from policy names appearing in evaluation metadata
- **Context-aware user messages** — each denied topic gets its own friendly redirect message 
  instead of a generic "request blocked"
- **Input tagging** — random suffix per request for prompt attack detection

**What's next (Day 6):** Contextual grounding — adding citation and hallucination checks 
so the model's responses stay anchored to source documents.